## Exercises and predictions (draft)

Structure can be given later, but these are a few ideas with some implemented:

* Exercises
    * Writing the formulas for computing the attention step by step in the way the UDL textbook does it. To check whether they got it correct we just compute some examples and they have to check whether they're results coincide with what we have computed (like in the lecture's theory notebooks) (1e)
    * Why is the MLP the floor? Explain in your own words and then check with the explanation below.
    * Description of 3 scenarios and ask the student to classify between transductive and inductive setting (probably there are some tricky examples that we can look for so that the student is lead to error through his intuition, if possible). The last one will be node classification, so it segways nicely into it.
    * How to add noise and where? (Little comment: As Matteo suggested, we can ask them to create a function that generates random edges)
    * Can convolutional neural networks (CNNs) be seen as a special case of graph convolutional networks (GCNs)? Can GCNs be seen as a special case of CNNs? If neither is a generalization of the other, explain the main differences between them.
    * Is the adjecency matrix nilpotent? In an undirected graph? What about in a directed graph? Are there any types of graphs that induce a nilpotent identity matrix? (Little comment: when we explain GNN, we will have to explain adjecency matrix properties such as what A^k represents)

* Predictions
    * We take a toy example and try to reproduce figures like the ones in the paper of Brody et al. (section 3). We ask them to predict what the figures will look like based on whether we have used static or dynamic attention in the toy example.
    * Will GATv2 and GAT ever go below the MLP floor and at what noise level. Will they both collapse under similar levels of noise?



# GCN and Attention

This notebok invesitgates graph convolutional networks (GCN) and attention mechanisms. 

First we start by defining a graph neural network (GNN): a GNN is a model designed to operate on a graph data structure in the form of node embedings and an ajacency matrix. For each layer in the network the node embedings are updated allowing nodes to gain context from the graph (13.3 of UDL).

A GCN is a form of GNN which aggregates infromation from neighbouring nodes. There are many ways of doing this, one of which is attention.

### Noise

Structural noise is unwanted information in the graph structure, such as incorrect, unwanted or missing edges. In this notebook we introduce structural noise by randomly adding non-existing edges to the original graph. The proportion of added edges is controlled by a noise ratio (p). Noise is added to the graph used for training while the test labels and evaluation setup are kept fixed.

In [ ]:
# Codeing exercise on introducing noise here

### Attention

Attention is a mechanism for computing a distribution of weights over a set of neighbour embeddings (keys), given a query embedding.

The standard **Graph Attention Network (GAT)** computes an attention score for each edge between a node \(i\) and its neighbour \(j\). The scoring function is:

\begin{equation}
e(h_i,h_j)
=
\operatorname{LeakyReLU}
\left(
\mathbf{a}^{T}
\left[
\mathbf{W}h_i \,\Vert\, \mathbf{W}h_j
\right]
\right),
\end{equation}

where $\mathbf{W}$ is a learned linear transformation, $\mathbf{a}$ is a learned attention vector, and $\Vert$ denotes vector concatenation.

These scores are normalised across the neighbours of node $i$ using the softmax function:

\begin{equation}
\alpha_{ij}
=
\frac{\exp(e(h_i,h_j))}
{\sum_{j' \in \mathcal{N}_i}\exp(e(h_i,h_{j'}))}.
\end{equation}

The updated representation of node $i$ is then computed as a weighted sum of its neighbours:

\begin{equation}
h'_i
=
\sigma
\left(
\sum_{j\in\mathcal{N}_i}
\alpha_{ij}\mathbf{W}h_j
\right).
\end{equation}

where $\sigma$ is a non-linear activation function.

Here the ranking of the attention scores assigned to the keys is independent of the query, this attention mechanism is called *static attention*. This limits the ability of GAT to change which neighbour is considered most important for different query nodes.

**GATv2** modifies the order of operations so that the query and neighbour representations are first combined, followed by the non-linearity:

\begin{equation}
e(h_i,h_j)
=
\mathbf{a}^{T}
\operatorname{LeakyReLU}
\left(
\mathbf{W}
\left[
h_i \,\Vert\, h_j
\right]
\right).
\end{equation}

This allows the attention score to depend on the interaction between $h_i$ and $h_j$, enabling *dynamic attention*. Consequently, the same neighbouring node can receive different relative importance depending on which query node is attending to it.

This notebook will explore the performance of *static attention* and *dynamic attention* in *transductive* node classification as graph noise increases. A *transductive* model predicts labels for previously unseen nodes within a graph that was available during training, whereas an **inductive** model learns a function that can generalise to new, unseen nodes or graphs.

### MLP

As a baseline, we also use a Multi-Layer Perceptron (MLP). Unlike a GNN, an MLP does not use the graph structure or information from neighbouring nodes. It instead makes predictions using the features of each node independently. This provides a useful baseline for determining whether using graph structure improves node classification, particularly as structural noise is introduced.

### Sensitivity Check

A sensitivity check tests whether our results change when an alternative experimental choice is made. This is important because it helps determine whether our conclusions are robust rather than being specific to one particular dataset.

To assess this, we repeat the experiment using the **CiteSeer** dataset and compare the results with those from the main dataset.

In [1]:
# @title Setup (run first): installs the packages and loads the interactive and the questions
%pip install -q numpy ipywidgets
exec(__import__("zlib").decompress(__import__("base64").b64decode("eNrVPNty28aS7/qKOXBlBdggBFISI1Ei6yi2j+2yk9hHTuwKl6UCiQEJCwQoALyZpap9Oh+wH7G/sA/7tp+SL9nungEwAEhK3mR3a+XYAubS09Pd0zdMx+Ueu3G554f8xkludaNzwODHn86iOGWTdBocUMNz1mWuP0r1mTPmXe2JN4I/Q81kMyfkATbQDzT44S282kP8I15b8H7aOm2ecHifzlPuQsPZ+dn3Z03NJOj5zzCKXR53tXg8dPRm0xT/2VbTNmBuAFhu6TvDPmc04mEKgFvO92duG1qSiPA69r73XM0Qu7i+hn14mqZdJuk64L0D685pxFGUbjZDZ3Q7jqN56HbS2AmTmRMDwL8ISjhheiFw69hK08xxXT8cd5qt2UppPoB9rBqJ/xX7xLQGtCgjlr6bTjpN2/5OaZw6q4boOLft2eoiWvDYC6JlZ+K7Lg9LC3hRmDY8Z+oH606yTlI+bcx9s+HMZgFviAZTu+bjiLNf3mjm36NhlEZmAvtqJDz2vYtRFERxZ/O8fwgcOhzcA8xG48sMERjzNGnU+ql37jdoZept1roDZx3NS70kHlv7W6X++/ucFcwSODRQ+mBsmAIfNhtkfmPC/fEk7TSt063EvHD9ZAaLdIZBNLothkjogTPkwWZDOwDucOCbdQqEpoalAN227TJtWoj91InHftgBNjObncxWEuBdCdgJAasRTswFAUjTaNpp55NprylfpSBoTkn+VMIo+6xCrstmE/ADsfddhqOQYuVhB/KANWLH9ecJ0Kwkt7k4QzOryLSyz2PcZ7VLyqIfTkC80tKaZc6dqKyLOYEMo5DXuFWiT8eLRvMEqCTwL0ghzn15myBjuOYesNaX+Wyd4rGcA1fCHK6ky1lpf3SYJ44LJ7EMsSo4e1bA9mQ+nPppjdM1ZorNoUZ9FPIF6A6IvzMMuFtbg7RuhUgzZ+Sn606ztsjEx/NWOyUFzTNo2bGwSVgeQDHmabz+g2Keye+Dgq4y5vRBxhBqnQnq2y0SJnVAjUywfm070IZ0kTAC7qWd4xKqubhe1CSucvxAy2R6Xjl2dfWy1dQoOhGwvTwS1g7snrDmT0DXNxrsbs6T1I9CNnTC2w77IF/10GDJJFomxYCQJnzLD63z4ZeX1x/f/PwTWt1NbumbHeFKZNC7Wj9belB1COAHiDeXw66dKdi3vGWgGWL4vdiWi84MWg0d1YZ0ZfAHGDyPQ3JnLJ6MnBkXI6yYg7kYcV375xA8Bu1yGPfAVciBKRQpgAEVr4E6CnE6zAnhv2TJY7T88Ogyh13TmWRCzFg6cVLAY8GdIIEX2ER5LxYyJ1vCi6Mpe/N+nU6gQ9q0zCd7/fHHdyaTjfmMDBkzBwgkz8nfDwd9LRuiDcxKTzZFG+Tw8ECUOCFX92dr6SEwJ2Gf8iF8NeKzlL2hUS/jGGS0NF3iqyP2ugeE7mW0vTwa9pDuG8G5DEvjHhsv/d6bMEmdIFBXTqOM2mBsuHV55APXjNJ6kuM/gb4uaAQk+WQhBj866UT3Di9df8FGgZMkXS1zDzQFMejuVcbcaVsQpYGHBQIoA7jUR2m8dBKySRSQb/sJ7CNn4AjF6i5+/5d/Qw+XHKTuJ+sdPeh0jLsanmPoFTYUXlFDqDsW6p/W/IHETXdBzGN/Js6NkMTd4J15GtXBWaCObmjnupabGK0YhWYip6iuKT2kT3di8/s//oOxjzDAGTt++ABWuaR3NbS9Kpa0SgVJalMwAclWUayvVYZuMpXghtQEmTaIwhtBBP3GqBwOj4VgWoDv1sIJ5txKUtiuXhmVEU2MAbwOLyHKCBXpwl6tVxcQz4+TFOQch/cOa0CFsB986zrPoykGOaWVln462aaeGByNaLkdBSkumfMBC36M53zrkEIOYJSUS/AltINyELjaDywK8q3tOsWMLF5Xk853Gs3Q89Z6FQsij26NovlPDTzMzFVABmWLCsjwFPKW6/BuWSMVYrx9XFX8aGxN+nKxQ5ruJeXfwPjwB3i3e8w25mm7pa6Cisq1LV01AogT+RhSyZFV9QX0GgX+6FbPz21Vd6gjqEUZgK4iao5ff4hWel9qkOfX14bJ7kykq4lqhToFcJN2bwrQg7qmcQJ/HN7AyZ6CIKETyGNMh5B8drU2RZbwB7QO2fBBGZeqnoMmRc1l1hWbjYNtNvBAacgM3MHBQfYIWy1lgQ5cHqgtMJScKyfFYBwm3Pi4AWeU+guug4FO/VHRCUp7HUI8qDRtySsp7+F8OlujQxHODr7N/6Hhb0324r3Jrt/9/P4l7KRpm6zZNplttXac6icwht3ydWLiA1jy2OfwDAj4IfubP56DVmyyyGM/xJG7ZjxlTmD9b+bBxrGPnbzJbfdcMx/Iiw0dHOzax8PjoZSKH39+8fIdut39HLCujR10ArRXVx/xV5VrmDKrMRJwGLbPjk9UBPuwy1PvmOM24HHojkZD8dh23LMzTZ0Fj8Pzk3ZTDPjeaY1sTxsYpoLVoiWREg810dG2iBMCyxJ9ZcSO4Y9Ni41c3vLEumftYZt79HgM9Oan9NgcnY4cMcB2j9vtYY7YIAuT0EQGHKzx0AdTmaJdHsfObNJhEIfNp2ECQmadM7SiKE7sFl5vz828twnvTdEvpYzdQdPduSUX+BlTI6BqYGexD6ECTzoIid3e+BRG4Kw1u4O3ZIJQ8nHMZ/oLEEQghhOv30XR7XxmluSVnVhNQ6zzAgThLT19hqdwZo2icOQALeGv3od3vub6W1A8+SMoMGflJ92mEKcp6AAx8yuPo0TXW+wpHjv6ZRRj+h1ofNsZ4Cm0bLb/54ncXQhT14x466JvD0EStCIVpkUolrNe94D7S8UEvheIOYkTx6ADvVD/DPubrXXDJJzyl2Vfi6Z87GAABM9DnmaPs4mPT6RAYKCbrme86wWRo2hT8PHeW8CEGWd/Ab+jRICyPY4dP+HsV7R1FAlBvCM0L9hXAWAjId2bEDjN+AhUAduUQd5rNUX+PqOu9XEvVWMM3buFWsuEsUsyWsoApIBr6Idjposcs9C8STqH5zQxOiX5i+aBq7ApE9Od0T+yLYh5MNe/1mNx4NgS4x79K+sxUNhfM/39lH1VInD30QBA3jIOFtOTyEunzsrWrxUAL4XAAOX1a9Zg1xaM0GHyLecz1wfzjB6nUSP/S3bEXlrJfFofWywI2sFN9Fs/dMtC+tbDeNyTK9MRy/uiKahwDiZkAt0PCKkqjrgK64L/g3q9LIKfQVA+fwBwwM2/4uLPaAV6+yDeSuN/Q1d6Nul3XrxH6forADCs2FnwQDf6HZN8iAFMwzEv3nfkmA/FGBxgss6gBBVPZs4BwcjfKnH6K1z4PXAhJ4sBZH4LUuBmE8p5hBEcT3RTXxEnmnBaxZNdGYcUcIkEcluAsDvK8URq006w2d2OvuQ7mXviSRegEjlh+88IOlLT+ojsS52uDt0SK9EtXwoKmjsDDWBut6qXaTHcMC3kuoMCULFZ4lw0xZ7OiwFhR8NUponuF7TZD14OBTdMoqCOzwG/BsA1FpQ4iizzQ9yjNjEnX/xG74uPntAkkwqTvVbYvYPVhRP7GzGrKnbZPp6yV/3sqS4bdV5Vqen+RuxoGZKDIB6/ZdwSbAShaBrbWUT8leP1JuholakZ8zJifPEFNSZAjFdIA4VtuaIgzSsVRZLyWdI9tkGvBHHXtuxTVd2BdiatEYM/EE0hDPOceZDeQLtu5Drfi3kyQf8NPGRw54UJjechOrEYzo94EDxSdywzF1fQEdaxwiieOhDv+l95V0dX+wVGKUSUwiXAdhD2HTKONKqBQlMnp4EoAKjkLk4BkCI1HKLRb8ePAP85OD6A3xRT22End41uAv+W6wuDeVHMwFNZYEixtCju0437fN5i67xVPm+1Yx52p9gH4jDmKIskPXCOm2Cu687AletMGeWfQQhgnaQBlj0GjwglQ0PDLiw6ZVXBnJcjbhS9skkrdROuhGg90zXth+gG2uAePxUvz+ClCS9jeKkNX+TDz3HCIptg29kU9vQpa9XmLbGngaoKhum0zhHTm6BmaOWnLCXe6RnzFuUR5/kYICBvnNU0ylIcWYyMeCZ4V90Nem5xDL/FBDrNlD6hAXCouuKA0/kCOVy0sgaIcqT4IPXAiTLZjck8CHA4/qK3G6SpCOE6ZXYu8TOCnzA3dpYheHZTiCjIZ6OcnfTbDpk3Dyk0SHYn9K8A2bJTLTfRB5wGxrb8JnrZQYCk9BMP/EZQq1fGliRnzQEu/N+fnJ8YbNsPPa2SLEMC97UrbUDrA3JX1Y8LL+kXZisgUK8oBTkdeFIA8LQNkfW+wzbo0evcsG5usOXmBtv4vVbo49iZzvQkjWbgMKufjsB5TmSAQeFFv++HqZ70/Y4PItNCm9EWh9ZHpuGBPDbZqTGgNkooENTBjrACs0xTkAv0QVPMWdgG+bIGinPAQ4GTAdLaVGIROQlRSWF8aVyrGDeCcbiBvo9Wk2ReT+Efn+Rd9KDgU6/oqMq/9uQ7u7XK/mrsO5bOISSmpem7J+g6sX/SdCPVxK1SfWUyEHCgKSlTzMzQB8zu875GGRD0bZ1wNIEmbQohSEBJePGd48S2624/Jnzx2x1bdbXNqmM1vXuNreF5LZ/zL6bQhr+wyQ8CeKOV4RWnN7JFN+IBmutZYO9Q+Y4MQ8XDvdbbJPeXRwgGU7407fpXcm8OL5PFmC18vvwhAgQxo7dZ3rPNBOCrnxay9HTpvorGVtMgTLraJE1nnaOj5XJpLY+tKB4ftWzbPgLQFSwPlasXXW3XPaDsGlD9FpCW44/8mnAHHLuZfmUyeRDuFPq/Mxn4TLjLYxtDrZb4t9227JK8vSo5dRGmoIA4loc2NdWXws2ZdFvnLWNQsmpfCqtWjagjCzYERkpHkXqHLu0X+Mc6xUMC/vFHkNzvQXlqt5sv9+onItwFahLWE5YFhl9RtNesRnCqjoVZqJf1xLjIFfurq48sxfRNuoxEGK2i7u9GPeH4BcpHX+tu354arI0bga35xdbg6dg6xZ3dbfx7TTk8fnirDVAvI3z0kZhyonZ/zsikHtYtztn3tl2ChOeubuf3sKe0HTifMR+J84ms+iI30lTOKu7Tr7TL47EZoQqTbXs/zGRfQ0szYli2lZ94Valf9X2TfUHbL0TAMOAg7//yk/ppwHtIevb7P/6VoXSB2ZCAOtaxh1qAxlwe4Z6rH4BysuW5CkQFlLHRtwf7STjy41EAHltGRVXg5VZHGSVVicnI0NWOrfYDBJREAsGh5LU2uEe9FEe3XLSSjOVtjUx/Wccam0WU/29AnAH+hvx6eqTuf4c8NGpycEdy0KjJwSs0r7JNYfWzEquPt24x2xph9agttXcgr12i1u3Vc3GaZn0BGuiRoj/pcA6dOJO4BYiZyca+W9ainwoVWlKeO1Xl8YmiKVW6wqJJ7xJvPznxK7xIBMxg+OFgA4uidAO1MkQPKrKNOEII4CUc6eqjS1zxNjpWyyNSRTNxGwrpf4+EEjoPtBHKNg/nUx5jiC3mlo/AM3Z4eVTGEA4LIf6gsAghOSlE4pMiCk07lwDB7Hkc6E9o48YOoVAFQXxQQVnYznhSysChM5NpGIRoJn1WEiLUUr0WirlVo1MzVp+AuC0JS/2qIQFKrb0XgpgNDi3KlGDMToRKur1d0uX/PbkGkU50Rxof8OcAhZpfABi+zuW62c6l22TnLUXGh/hNo0sUeYSP0GydVXwE8jEx/4pICLJW0MEf/CiMquV15vQuYCiNO9ilb1FE2aopxW7VkmoXJW7dVLzLdUt5UQUKv9jVxGmrnQd7vi4M+2YB7BwjO8/LLKy7MXBU9dySOIpP40iDBlbFSqPAT1Jdmf2A8UY90BUDcJ3y93zoQSI6fTKcSMJSP15zymw8sfaZ+PWUWc2zgwfdg1XFFLwG6kwqZiCD1z6p2gLc8wQiJmO/s+AdltTEJj8x6PPg5snn0Z6MvBEfnWm7nYLcIZB+ABLlEU7APt8VN2dKUQVP6Dj3Yelw77bdqgOYb0ENqcqeXT6k7tmJ3j2+yANYA8Pg7wlJsiBIVTcp6PwZyiiGGTzW1cjkCg8HSX2RRBBXTRL1HKDqxLgZxKYvXUA6G1co+lcJppv6Td44V3IfazGFQnY8eiPuBzrBeQo7xLxSU/nuMHJiF1Hpl3UWpXiy9A79usl1KZlMJdlTPn3ccetXnBy/MdklozBy2NtQzoOuVarXvmCeF0Lo6sluusi19d4Spof5WmQuiqxKXUaG+K25y2ro4fDeRr1sW8vO5DemSkDrud4sUVXJDu1CpU6qhI+03lVmdDHnKVLj9F1T4LDZFvfeb6renBQZtOF+Y9zQQFwAFeN+t4v9CGTuNneSEhuyslf9u0HZ0N5XmENCpvqApRWwV6MdufcbJEr9ZpqSRynPJQvW22SHjhbazqetmwv4GM1WjzIgmWPdUhy2FvQpYU1bRDDtfcHKH4lRjnpCrbCPIpwn3jujifykrScoCE7A3CilE+jgd3O6qeFgpG9sw+afwmEyuxD//ue/q2+vsXgAZuJ3FkxwAjeFlwzASRXlfCCwoySpFqIBCf//FaL9H5WhZb1017NeqlEeFMNhyKvOtiz1P1bTVu8/pn48aLLkTTI9L3mTFWp7i09YbZbcYWXe7ooYWaqUg6kVQ/2ppWjlO2tYrrmnIu2RlVd70P+2Op4cSBj94jcglAN6mkUryEYCeyB4ZaJIJu4/lo/Dn5YeRWEI3uO26rCHsJ6APxbwbVjLrsdwNGMbFhkp9Uq15WscOy2pjm/YscCtM+SgJ1Xs1U7HS5H2WZp8D8hH1IyeqNVvmUpb+Ik/DB4BVcEQDW1ab8iHL4AKm82DS6AIFXvDtwv8pwFKE1pS3pAXtzpTPxRXlJpebLDS28XYmZHWv1Buup+JYjzpD+zk/vaqUdFYr1CjRWoVoheAjbQVmU6ZFHvyAr66oIvPDfqQ3RH3ngnps6zwVlS3battkwC9sFyNuL0WUTVFcACmURiB6Rxx83kUwu6cxMybJFxwx/5IMTDRwGatnNjCA9qDaxlAxqqLx1Crna8CTvRmo0BANEp82cFWx91edVgjQLm4/AFSXiwngGuDnjuzGDyw2JltKzksfX/eU0y2r5CsVESGLtSzuhtKN+N7Gxkm2kVpxp578defTPYZAtbPTUphwdMZJbDOT3bcYX3ChH4Vfq7JhLZiyC+6owtiQTWAKTtn+kymnYTfiTUFb8L0mubr5Bp2YS04RvTLWXXfYvZVXOvoNrGegIx7l+oyMDaA4CGcR/PkZj5zQUvQ96vduYLtxVUQrV9/up+tlAoEO8s4kQ9SVE0JAsvvlWq1+86zIi+9Z2IeC/XbwvKbD0XYJUOS4XwIurFYbg++wO4Tu4I0FU00WqIejjiDz4ycZuHuJ0BoIVkhRedyHa+0r1mU+BiTdWIOmtdfcOkYCypdZCakXT4vzXJFbpZ+2ZFC8mQZVnVJZ0h1RPyCyoU3n3GLF+T5Yz5UxABoDj7rDbC1htazdxaCPX6R5kOLbEgMK3mCbWL2bQKW53GmYB5vBPPVOjlMt3y28bMQSoosGDpiOmFDdyI+Y0L3s5J2EUBqFWEPczaTp4y9rRMU0b3x/GNou8JvVDYlJUkZ0//2wd5D7It9Syq2OztXxf8f4KJiaqrnMTcLcCRQ1+9dqGQZqFjubqNwYWvKqMRFagERKA5ZroX3V7YapfweXm3Sn6pFbvVV5Eo507OsYIFuFmhHeNstkAefyrVIv5l5aZeAawqBM0lLDIxH1299mSep761vpMuZ90hdmi2vlm8JZ1FTjIIVDSEaXnBdbN5kGm1BDtleigbmzzBz+CZSY1CMr6ynlIvVS8WkOcSGA1nntb3MS3IETPqbohWr+fFyxCJhVMLDlBqdeYKlBdWiIjKNtYqe4hYbX/F45Cc8r4gvuQ4eFTI8quRM9Sd+cqa84k3MYrzDpP19HpaWFSkc/XFYG7I0V9vuXeD7jpq5p7AR4+C/AKUg25M=")).decode())

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Exercise (1e): attention step by step, UDL style.
# The interactive below calls static_attention and dynamic_attention, so implement these first.
import numpy as np


def leaky_relu(z, slope=0.2):
    """Elementwise LeakyReLU. `slope` is the gradient for negative inputs."""
    return np.where(z > 0, z, slope * z)


def softmax_cols(S):
    """Softmax applied separately to each column of S (numerically stable)."""
    E = np.exp(S - S.max(axis=0, keepdims=True))
    return E / E.sum(axis=0, keepdims=True)


def softmask(S, mask):
    """UDL's Softmask[S, A + I]. Hide the non-neighbours, then softmax each column."""
    ### BEGIN SOLUTION
    # Step 1: wherever mask is 0, replace the score by a huge negative number (exp of it is 0)
    S = np.where(mask == 0, -1e20, S)
    # Step 2: softmax over each column, so the attention of every node sums to one
    P = softmax_cols(S)
    ### END SOLUTION
    return P


def static_attention(X, mask, omega, beta, phi, slope=0.2):
    """GAT (Veličković et al., 2018). Returns P, where P[m, n] is the attention node n pays to node m."""
    N = X.shape[1]
    ### BEGIN SOLUTION
    # Step 1: transform every node,  X' = beta 1^T + Omega X                      (D' x N)
    X_prime = beta + omega @ X
    # Step 2: score every pair,  s_mn = LeakyReLU( phi^T [x'_m ; x'_n] )          (N x N)
    S = np.zeros((N, N))
    for m in range(N):
        for n in range(N):
            pair = np.concatenate([X_prime[:, m], X_prime[:, n]])
            S[m, n] = leaky_relu(phi[:, 0] @ pair, slope)
    # Steps 3-4: mask the non-neighbours and softmax each column
    P = softmask(S, mask)
    ### END SOLUTION
    return P


def dynamic_attention(X, mask, omega, beta, phi, slope=0.2):
    """GATv2 (Brody et al., 2022). Same inputs and output as static_attention, other parameter shapes."""
    N = X.shape[1]
    ### BEGIN SOLUTION
    S = np.zeros((N, N))
    for m in range(N):
        for n in range(N):
            # Step 1: concatenate first, then transform,  z_mn = Omega [x_m ; x_n] + beta     (D')
            z = omega @ np.concatenate([X[:, m], X[:, n]]) + beta[:, 0]
            # Step 2: nonlinearity first, then phi,  s_mn = phi^T LeakyReLU(z_mn)
            S[m, n] = phi[:, 0] @ leaky_relu(z, slope)
    # Steps 3-4: mask the non-neighbours and softmax each column
    P = softmask(S, mask)
    ### END SOLUTION
    return P

In [3]:
# Run to see the static vs dynamic attention interactive
attention_interactive()

**Interactive 1.** A complete bipartite graph of "query nodes" $\{q_0, \dots, q_9\}$ and "key nodes" $\{k_0, \dots, k_9\}$, where every query attends to every key. The left panel uses `static_attention` (GAT) and the right panel uses `dynamic_attention` (GATv2). Each grid shows the attention of every query over the keys. The dot marks the key that each query attends to most. The bars show the attention of the query picked with the slider.

**What we are trying to do.** Every node carries a feature vector in $\mathbb{R}^{10}$ that is one-hot. Key $k_i$ and query $q_i$ both get $\mathbf{e}_i$, the vector with a 1 in position $i$ and 0 everywhere else. The feature of a node is therefore just its label. The only way for $q_i$ to recognise its own key is that $k_i$ carries the same feature. In the notation of the exercise this gives $\mathbf{X} = [\,\mathbf{I} \;\; \mathbf{I}\,] \in \mathbb{R}^{10\times 20}$, with the 10 keys in the first columns and the 10 queries in the last ones. The mask lets every query attend to all 10 keys and to nothing else.

The task is for every query $q_i$ to put its attention on the key with the same feature, $k_i$. This is a small version of the DictionaryLookup problem of Brody et al. To get there, each model starts from random $\boldsymbol\Omega$ and $\boldsymbol\phi$ with $\boldsymbol\beta = \mathbf{0}$ (using $D' = 16$ and LeakyReLU slope 0.2). It is then trained to maximise the attention that each $q_i$ pays to $k_i$, using 300 steps of Adam on the loss $-\frac{1}{10}\sum_i \log P_{k_i q_i}$. The grids and bars show the attention that your `static_attention` and `dynamic_attention` compute with these trained weights. Re-run the cell to train again from new random weights.

**What we get.** GATv2 computes **dynamic** attention, so every query can rank the keys differently. It solves the task, with each query $q_i$ attending to $k_i$ and the dots lying on the diagonal. GAT computes **static** attention, so the ranking of the keys is global for all queries and does not depend on the query node. It cannot solve the task. Every query attends most to the same keys, so the order of the bars stays the same as you move the slider.

**Why GAT shows ties.** The definition of static attention only asks for a highest-scoring key $j_f$ with $f(q_i, k_{j_f}) \geq f(q_i, k_j)$ for every query $i$ and key $j$. The inequality is not strict, so several keys can share the top score. Here every query wants a different key, so training cannot favour one key without hurting the other queries. The best GAT can do is give several keys the same top score, which are the same keys for every query.

In [4]:
Question(1)

**Adding more questions (for the authors, delete before release).**
1. Put a code cell `Question(n)` where the new question should appear, with `n` the next free number.
2. Ask an AI assistant to add question `n` with your question and sample solution, pasting the instructions below.

> *Instructions for the AI.* The question text and the sample solution live in the first code cell (the setup cell). That cell holds all hidden code as a single line, `exec(__import__("zlib").decompress(__import__("base64").b64decode("...")).decode())`. Base64-decode and zlib-decompress the string to get the Python source. Find the `QUESTIONS` dict inside `_define_ask` and add an entry `n: dict(question="...", solution="..."),`. Compress the edited source again with `zlib.compress(..., 9)`, base64-encode it and put it back in the same place. Change nothing else in the notebook. Run the setup cell and the `Question(n)` cell to check that it shows.